# Bronze Layer Ingestion

This notebook reads raw parquet files from the `source_systems` volume and loads them into Bronze Delta tables.

**Key characteristics:**
- **No transformations** – data is stored exactly as it arrives
- **Incremental** – only new (unprocessed) files are loaded
- **Scalable** – add new files by appending to the config list

In [0]:
# ── Configuration ────────────────────────────────────────────────────────────────

CATALOG  = "nyc_taxi_lakehouse"
SCHEMA   = "bronze"
VOLUME   = f"/Volumes/{CATALOG}/{SCHEMA}/source_systems"


In [0]:
# ── File Registry ────────────────────────────────────────────────────────────────
# Each entry: (sub-folder/filename, target delta table name)
# To add new months → simply append a new tuple to the relevant list.

YELLOW_FILES = [
    ("yellow/yellow_tripdata_2024-10.parquet", "yellow_tripdata_2024_10"),
    ("yellow/yellow_tripdata_2024-11.parquet", "yellow_tripdata_2024_11"),
]

GREEN_FILES = [
    ("green/green_tripdata_2024-10.parquet", "green_tripdata_2024_10"),
    ("green/green_tripdata_2024-11.parquet", "green_tripdata_2024_11"),
]

FHV_FILES = [
    ("fhv/fhv_tripdata_2024-10.parquet", "fhv_tripdata_2024_10"),
    ("fhv/fhv_tripdata_2024-11.parquet", "fhv_tripdata_2024_11"),
]

FHVHV_FILES = [
    ("fhvhv/fhvhv_tripdata_2024-10.parquet", "fhvhv_tripdata_2024_10"),
    ("fhvhv/fhvhv_tripdata_2024-11.parquet", "fhvhv_tripdata_2024_11"),
]

ALL_FILES = YELLOW_FILES + GREEN_FILES + FHV_FILES + FHVHV_FILES


In [0]:
# ── Helper Functions ─────────────────────────────────────────────────────────────

def table_exists(table_name: str) -> bool:
    """Check if a Delta table already exists in the catalog."""
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    try:
        spark.catalog.getTable(full_name)
        return True
    except Exception:
        return False


def load_to_bronze(file_path: str, table_name: str) -> None:
    """Read a parquet file and write it as a Delta table — no transformations."""
    full_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    source     = f"{VOLUME}/{file_path}"

    df = spark.read.parquet(source)
    df.write.format("delta").mode("overwrite").saveAsTable(full_table)

    record_count = df.count()
    print(f"  ✔ Loaded {record_count:,} rows → {full_table}")

In [0]:
# ── Main Ingestion Loop ─────────────────────────────────────────────────────────

print(f"Starting Bronze ingestion from: {VOLUME}\n")

loaded  = 0
skipped = 0

for file_path, table_name in ALL_FILES:

    if table_exists(table_name):
        print(f"  ⏭ {table_name} already exists — skipped")
        skipped += 1
        continue

    load_to_bronze(file_path, table_name)
    loaded += 1

print(f"\nDone — Loaded: {loaded} | Skipped: {skipped} | Total: {len(ALL_FILES)}")